In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dask_image.imread import imread
import dask.array as da
import os
import numpy as np
import joblib
import dask  # Import Dask first
dask.config.set({'dataframe.query-planning': False})  # Disable query-planning

import dask.dataframe as dd  # Now import dask.dataframe
import pandas as pd
import dask.dataframe as dd
from spatialdata import read_zarr

c:\Users\matti\.conda\envs\ilastik_napari_182\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


In [3]:
from spatialdata import read_zarr

sdata=read_zarr(r"C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr")
sdata

c:\Users\matti\.conda\envs\ilastik_napari_182\Lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr
├── Images
│     ├── 'channel_0': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_1': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_2': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_3': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_4': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_5': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_6': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_7': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_8': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_9': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_10': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_11': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_12': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_13': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_14': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_15': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_16': DataArray[cyx]

In [128]:
from enum import StrEnum

class Statistical_Functions(StrEnum):
    SUM = "sum"
    MEAN = "mean"
    COUNT = "count"
    VAR = "var"
    KURTOSIS = "kurtosis"
    SKEW = "skew"
    QUANTILES = "quantiles"
    RADII_AND_AXES_MASK = "axes_mask"

    @staticmethod
    def get_values(args: list["StatisticalFunctions"]) -> list[str]:
        return [stat.value for stat in args]
    
    @staticmethod
    def get_single_stats(stats: list["StatisticalFunctions"]) -> list[str]:
        aggregate_stats = {Statistical_Functions.SUM, 
                    Statistical_Functions.MEAN, 
                    Statistical_Functions.COUNT, 
                    Statistical_Functions.VAR, 
                    Statistical_Functions.KURTOSIS, 
                    Statistical_Functions.SKEW}
        result = []
        for stat in stats:
            if stat in aggregate_stats:
                result.append(stat.value)
        
        return result



In [163]:
[i for i in Statistical_Functions]

[<Statistical_Functions.SUM: 'sum'>,
 <Statistical_Functions.MEAN: 'mean'>,
 <Statistical_Functions.COUNT: 'count'>,
 <Statistical_Functions.VAR: 'var'>,
 <Statistical_Functions.KURTOSIS: 'kurtosis'>,
 <Statistical_Functions.SKEW: 'skew'>,
 <Statistical_Functions.QUANTILES: 'quantiles'>,
 <Statistical_Functions.RADII_AND_AXES_MASK: 'axes_mask'>]

In [144]:
from harpy.utils._aggregate import RasterAggregator
from functools import reduce
from ilastik.napari.object_classification import check_and_convert_arrays_to_dask

mask = sdata['masks_whole']
images = da.concatenate([sdata['channel_0'],sdata['channel_1']])
annotation = sdata['annotation']
stats = [stat for stat in Statistical_Functions]

mask = check_and_convert_arrays_to_dask(mask)

if len(annotation.shape)>2:
    annotation = annotation.squeeze()
annotation = check_and_convert_arrays_to_dask(annotation)

images = check_and_convert_arrays_to_dask(images)

# start workflow
images=images[ :, None, ... ]

mask = mask[None, ...]

if mask.chunksize != images.chunksize[1:]:
    mask = mask.rechunk(images.chunksize[1:])

aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=images)

raa = aggregator.aggregate_radii_and_axes(100)

raa

,0,1,2,3,4,5,6,7,8,9,10,11,cell_ID
0,2.439437,1.564841,0.0,0.0,-0.134019,0.990979,0.0,0.990979,0.134019,1.0,0.0,0.0,1
1,2.763416,1.501450,0.0,0.0,0.163683,0.986513,0.0,0.986513,-0.163683,1.0,0.0,0.0,2
2,2.975000,1.958264,0.0,0.0,-0.073496,0.997295,0.0,0.997295,0.073496,1.0,0.0,0.0,3
3,4.093265,2.402311,0.0,0.0,-0.092717,0.995693,0.0,0.995693,0.092717,1.0,0.0,0.0,4
4,5.219499,2.845574,0.0,0.0,0.225241,0.974303,0.0,0.974303,-0.225241,1.0,0.0,0.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
669,5.062693,2.637059,0.0,0.0,-0.152949,0.988234,0.0,0.988234,0.152949,1.0,0.0,0.0,670
670,3.811477,2.551571,0.0,0.0,-0.100057,0.994982,0.0,0.994982,0.100057,1.0,0.0,0.0,671
671,5.756720,2.525367,0.0,0.0,-0.077121,0.997022,0.0,0.997022,0.077121,1.0,0.0,0.0,672
672,2.311644,1.902516,0.0,0.0,0.603026,0.797722,0.0,0.797722,-0.603026,1.0,0.0,0.0,673


In [ ]:
def feature_extractor(
    self,
    mask: da.Array,
    image: da.Array,
    stats:tuple[Statistical_Functions],
) -> dd.DataFrame:
    # feature extraction
    mask = mask[None, ...]

    if mask.chunksize != image.chunksize[1:]:
        logger.warning("Mask chunks and image chunks are not the same. Changing mask chunks...")
        mask = mask.rechunk(image.chunksize[1:])

    aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=image)
    single_stats = Statistical_Functions.get_single_stats(stats)
    features=dict(zip(single_stats,aggregator.aggregate_stats(stats_funcs=single_stats)))

    if Statistical_Functions.QUANTILES in stats:
        quantiles = aggregator.aggregate_quantiles(100)

        for i in range(len(quantiles)):
            quantile = quantiles[i]
            quantile.columns = [f"{i}_{c}" if c!='cell_ID' else c for c in quantile.columns]
        
        features[Statistical_Functions.QUANTILES.value] = reduce(lambda left, right: dd.merge(left, right, on='cell_ID', how='outer'), quantiles)

    if Statistical_Functions.RADII_AND_AXES_MASK in stats:
        features[Statistical_Functions.RADII_AND_AXES_MASK.value] = aggregator.aggregate_radii_and_axes(100)


    for key, feature in features.items():
        prefix = key+"_"
        feature.columns = [f"{prefix}{c}" if c!='cell_ID' else c for c in feature.columns]

    res = reduce(lambda left, right: dd.merge(left, right, on='cell_ID', how='outer'), list(features.values()))
    res = res.loc[res['cell_ID']!=0]

    return res


In [155]:
mask = sdata['masks_whole']
images = sdata['channel_0']
annotation = sdata['annotation']
stats = [stat for stat in Statistical_Functions]
stats

[<Statistical_Functions.SUM: 'sum'>,
 <Statistical_Functions.MEAN: 'mean'>,
 <Statistical_Functions.COUNT: 'count'>,
 <Statistical_Functions.VAR: 'var'>,
 <Statistical_Functions.KURTOSIS: 'kurtosis'>,
 <Statistical_Functions.SKEW: 'skew'>,
 <Statistical_Functions.QUANTILES: 'quantiles'>,
 <Statistical_Functions.RADII_AND_AXES_MASK: 'axes_mask'>]

In [156]:
from ilastik.napari.object_classification import check_and_convert_arrays_to_dask

mask = check_and_convert_arrays_to_dask(mask)

if len(annotation.shape)>2:
    annotation = annotation.squeeze()
annotation = check_and_convert_arrays_to_dask(annotation)

images = check_and_convert_arrays_to_dask(images)

# start workflow
images=images[ :, None, ... ]

features = feature_extractor(mask, images, stats)

features

[<Statistical_Functions.SUM: 'sum'>, <Statistical_Functions.MEAN: 'mean'>, <Statistical_Functions.COUNT: 'count'>, <Statistical_Functions.VAR: 'var'>, <Statistical_Functions.KURTOSIS: 'kurtosis'>, <Statistical_Functions.SKEW: 'skew'>, <Statistical_Functions.QUANTILES: 'quantiles'>, <Statistical_Functions.RADII_AND_AXES_MASK: 'axes_mask'>]


,sum_0,cell_ID,mean_0,count_0,var_0,kurtosis_0,skew_0,quantiles_0_0,quantiles_1_0,quantiles_2_0,...,axes_mask_2,axes_mask_3,axes_mask_4,axes_mask_5,axes_mask_6,axes_mask_7,axes_mask_8,axes_mask_9,axes_mask_10,axes_mask_11
1,0.014940,1,0.000340,44.0,4.177080e-07,3.515966,2.004929,0.0,0.0,0.000000,...,0.0,0.0,-0.134019,0.990979,0.0,0.990979,0.134019,1.0,0.0,0.0
2,0.285948,2,0.006216,46.0,5.530315e-05,0.635498,1.260174,0.0,0.0,0.000139,...,0.0,0.0,0.163683,0.986513,0.0,0.986513,-0.163683,1.0,0.0,0.0
3,0.065575,3,0.000950,69.0,2.238899e-06,6.491301,2.495530,0.0,0.0,0.000000,...,0.0,0.0,-0.073496,0.997295,0.0,0.997295,0.073496,1.0,0.0,0.0
4,0.033986,4,0.000301,113.0,6.067106e-07,10.932738,3.240001,0.0,0.0,0.000000,...,0.0,0.0,-0.092717,0.995693,0.0,0.995693,0.092717,1.0,0.0,0.0
5,0.192600,5,0.001088,177.0,1.364232e-06,0.980605,1.154426,0.0,0.0,0.000109,...,0.0,0.0,0.225241,0.974303,0.0,0.974303,-0.225241,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670,0.087337,670,0.000542,161.0,8.437232e-07,3.090276,1.913735,0.0,0.0,0.000000,...,0.0,0.0,-0.152949,0.988234,0.0,0.988234,0.152949,1.0,0.0,0.0
671,0.037531,671,0.000321,117.0,5.430691e-07,4.402264,2.351323,0.0,0.0,0.000000,...,0.0,0.0,-0.100057,0.994982,0.0,0.994982,0.100057,1.0,0.0,0.0
672,0.122221,672,0.000711,172.0,1.390167e-06,4.003164,2.057298,0.0,0.0,0.000000,...,0.0,0.0,-0.077121,0.997022,0.0,0.997022,0.077121,1.0,0.0,0.0
673,0.018110,673,0.000362,50.0,4.575726e-07,2.703542,1.860595,0.0,0.0,0.000000,...,0.0,0.0,0.603026,0.797722,0.0,0.797722,-0.603026,1.0,0.0,0.0


In [157]:
features.info()

<class 'pandas.core.frame.DataFrame'>
Index: 674 entries, 1 to 674
Data columns (total 28 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sum_0          674 non-null    float32
 1   cell_ID        674 non-null    int32  
 2   mean_0         674 non-null    float32
 3   count_0        674 non-null    float32
 4   var_0          674 non-null    float32
 5   kurtosis_0     674 non-null    float32
 6   skew_0         674 non-null    float32
 7   quantiles_0_0  674 non-null    float32
 8   quantiles_1_0  674 non-null    float32
 9   quantiles_2_0  674 non-null    float32
 10  quantiles_3_0  674 non-null    float32
 11  quantiles_4_0  674 non-null    float32
 12  quantiles_5_0  674 non-null    float32
 13  quantiles_6_0  674 non-null    float32
 14  quantiles_7_0  674 non-null    float32
 15  quantiles_8_0  674 non-null    float32
 16  axes_mask_0    674 non-null    float32
 17  axes_mask_1    674 non-null    float32
 18  axes_mask_2    

In [ ]:
from ilastik.napari.utils import get_annotation

annotated_cells_id, annotation=get_annotation( array_1=annotation, array_2=mask)

X_train=features[ features[ "cell_ID" ].isin( annotated_cells_id )]
X_train = X_train.drop("cell_ID", axis=1)

print(type(X_train))
print(type(annotation))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def object_training(
        self,
        X_train: dd.DataFrame,
        y_train: dd.DataFrame,
    ) -> None:
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X_train, y_train)

        joblib.dump(clf, os.path.join(self.output_folder, self.MODEL_NAME))

see https://github.com/ilastik/ilastik/blob/54e17482cfe8c186a05b367450c4661792f7048c/ilastik/plugins_default/vigra_objfeats.py#L104 for all features ilastik extracts.

we probably want to support:

image + label:
- 'sum' 
- 'mean'
- 'var'
- 'kurtosis'
- 'skew'
- 'min'
- 'max'
- 'quantiles'

label:
- 'area'
- center_of_mass
- radii and axes

These are all implemented in `RasterAggregator`

In [ ]:
sdata["masks_whole"].data

In [ ]:
import dask.array as da

mask=sdata[ "masks_whole" ].data[ None, ... ] # (z,y,x)

image=da.concatenate([ sdata[ _image_name ].data for _image_name in [*sdata.images] ])
image=image[ :, None, ... ] # ( c,z,y,x )

In [ ]:
print(type(image))

In [ ]:
from harpy.utils._aggregate import RasterAggregator

aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=image)

In [ ]:
aggregator.aggregate_radii_and_axes( depth=100 ).head()

In [ ]:
quantiles=aggregator.aggregate_quantiles(depth=100 ) # gives you a list of dataframes
quantiles[2].head()

In [ ]:
dfs=aggregator.aggregate_stats( stats_funcs=("sum", "mean", "count", "var", "kurtosis"))

In [ ]:
dfs[0] #-> sum, for each object, and each channel in image

In [ ]:
sdata["annotation"].data

In [ ]:
from ilastik.napari.utils import get_annotation

annotated_cells_id, annotation=get_annotation( array_1=sdata["annotation"].data, array_2=sdata["masks_whole"].data)

print(annotated_cells_id)
print(annotation)

In [ ]:
# for simplicity first try implementing object classification only using mean intensity
features=aggregator.aggregate_stats( stats_funcs=( "mean"  ) )# retuns a list of dataframes, take the first on (mean intensity)
print(len(features))
features[1][ [ 0, 1, "cell_ID" ] ] # only take mean, and only the first two channels
features=features[0][[ 0, 1, "cell_ID" ]]
features=features[  features[ "cell_ID" ]!=0 ] # remove features for background

In [ ]:
def featuer_extractor(mask, image, stats):

    aggregator=RasterAggregator( mask_dask_array=mask, image_dask_array=image)

    features=aggregator.aggregate_stats(stats_funcs=stats)
    for index in range(len(stats)):
        prefix = stats[index]+"_"
        feature = features[index]
        feature.set_index("cell_ID")
        feature.columns = [f"{prefix}{c}" if f"{c}".isdigit() else c for c in feature.columns]

    res = dd.concat(features, axis=1)
    res = res.drop("cell_ID", axis=1)
    res = res.loc[res.index!=0]
    res["cell_ID"] = res.index

    return res

features = featuer_extractor(mask, image, stats=["sum", "mean", "count", "var", "kurtosis", "skew"])

In [ ]:
features

In [ ]:
X_train=features[ features[ "cell_ID" ].isin( annotated_cells_id )]  # train on these
# drop the cell_ID column
X_train=X_train.drop("cell_ID", axis=1)
X_train.head()

In [ ]:
annotation

In [ ]:
annotated_cells_id

In [ ]:
X_train

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, annotation)
y_pred = clf.predict(X_train)
y_pred

In [ ]:
# run on all data
y_pred_all=clf.predict( features.drop( [ "cell_ID" ], axis=1 ) )  # gives us a prediction for every cell
y_pred_all.shape

In [ ]:
y_pred_all[:10]  # this is a label for every mask

In [ ]:
cell_ids=features[ "cell_ID" ].compute() # cell_IDs
cell_ids
cell_ids[ :10 ]

In [ ]:
# now generate the relabeld mask efficiently
mask=sdata[ "masks_whole" ].data # relabel this

In [ ]:
import numpy as np
import dask.array as da

# create the relabeld mask as a dask array

assert cell_ids.shape == y_pred_all.shape

max_id = cell_ids.max()
lookup = np.zeros(max_id + 1, dtype=y_pred_all.dtype)
lookup[cell_ids] = y_pred_all 
relabelled_masks = da.take(lookup, mask) # maps each cell_id to its new label

In [ ]:
from spatialdata.models import Labels2DModel

se= Labels2DModel.parse(relabelled_masks, dims=("y", "x"))

sdata[  "predicted_labels" ] = se

sdata.write(
    r"C:\Users\matti\Documents\WERK\STAGE\VIB\output\object\object_sdata.zarr",
    overwrite=True,
)

sdata = read_zarr(sdata.path)

In [ ]:
from napari_spatialdata import Interactive

Interactive( sdata )

In [ ]:
# dummy code to explain the working of np.take
import numpy as np

lookup = np.array([0, 10, 20, 30, 40])

mask = np.array([
    [3, 1, 2],
    [3, 4, 0]
])

result = np.take(lookup, mask)
result